# Buyer GPT demo: what it can do, without opening the game

**Sell Anything, notebook 5.** This notebook runs the game's cat sale (`buyers/cat_market.py`, the same code the game
server uses): four buyers open, take turns asking, I play the seller and type the answers, every buyer moves their
offer with a reason, and at the end they picture the cat at home.

Every version is run with **the same cat, the same random seed and the same rule for the seller's answers**, so the
differences come from the model:

| Version | Training data | Status |
|---|---|---|
| B1 | 100 sales (format 4: budget on the card, last two rounds only) | ✅ demo below |
| B2 | 250 sales (150 new sales written under quality checks; no budget) | ⏳ to be added below |

⚠️ The seller's answers were written **after reading each question**, truthfully from the fact sheet below, not
prepared in advance. Because the seed is fixed, the same model and the same answers always give the same sale.

In [1]:
import json
import sys
from pathlib import Path

ML = Path.cwd().parent
sys.path.insert(0, str(ML))
sys.path.insert(0, str(ML / "tools"))
import random

from sellanything_ml.buyers import cat_market as C
from sellanything_ml.buyers import stories

B1 = C.BuyerGPT(ML / "runs/sft_b1/best.pt", "cpu")     # the CPU is enough: about a second a line
print("B1 prompt format:", B1.prompt_format)

B1 prompt format: 4


## The cat (only the seller knows these facts)

A cat made from seed 5002, which is not in any training sale.

In [2]:
cat = stories.seed_sale(random.Random(5002))["cat"]
for key in ("age", "breed", "home", "fixed", "health", "virus", "company", "temperament", "quirks"):
    print(f"{key:12} {cat[key]}")

age          13 years
breed        a Siamese
home         indoor-only
fixed        spayed/neutered
health       has some stiff joints
virus        tested FIV positive
company      ['has never met children', 'has lived with a dog']
temperament  calm and lazy
quirks       ['uses a scratching post, never the sofa', 'sheds a lot']


The seller answers from this list only: anything not on it is "I don't know", never made up.

## B1: openings and the first question

In [3]:
def show(m, start=0):
    for line, src in zip(m.log[start:], m.sources[start:]):
        print(("  " if src == "player" else "") + line)
    print("offers:", {n: m.offers[n] for n in m.cards})

m = C.CatMarket.start(B1, 4, seed=5002)
for n, c in m.cards.items():
    print(f"{n:5} {c['persona']:45} cares about {c['cares_about']}  budget ${c['budget']}")
print()
show(m)

Hana  lost an old cat last year                     cares about ['affectionate', 'healthy', 'gentle']  budget $95
Owen  breeds show cats                              cares about ['purebred', 'show quality', 'good lineage']  budget $250
Maya  loves rare, premium and pedigree things       cares about ['pedigree', 'rare', 'premium quality']  budget $200
Leo   wants things in excellent shape               cares about ['healthy', 'well-groomed', 'vaccinated']  budget $100

Hana: Hello. I lost my old cat last year.
Owen: I breed show cats, so I'm looking for a cat with top-notch qualities.
Maya: I love premium pedigrees, so I want a pedigreed cat.
Leo: I want a cat with under control and long Grooming, no health issues.
Hana: Is she healthy? I lost an old cat last year.
offers: {'Hana': 60, 'Owen': 120, 'Maya': 160, 'Leo': 70}


All four openings are readable and the offers fit the buyers: Owen, who breeds show cats, opens at $120 and
Maya, who collects premium breeds, at $160, while Hana and Leo open at $60–70. Leo's greeting already breaks down
("a cat with under control and long Grooming"). The first question is Hana asking about health, which fits a buyer
who lost an old cat last year.

In [4]:
answers = ["She is 13 years old. She has some stiff joints, and she tested FIV positive."]
n = len(m.log)
m.answer_text(answers[0])
show(m, n)

  You: She is 13 years old. She has some stiff joints, and she tested FIV positive.
Hana: A Stiff joints at fifteen means more vet visits, but I'm not the excellent health I'm looking for. Down to $55.
Owen: An FIV positive cat can never go into my breeding program, so I'm out.
Maya: A kitten of no known pedigree can never be the pedigree cat I collect, so I'm out.
Leo: FIV positive scratching up on a routine over a long Grooming means no health issues. I'll come down to $50.
Leo: Can you tell me about her health and if she has any issues in the past?
offers: {'Hana': 55, 'Owen': 0, 'Maya': 0, 'Leo': 50}


**What works:** Owen walks away on hearing FIV positive, for the right reason (an FIV-positive cat cannot be
bred), which is exactly what the training sales teach.

**What goes wrong:**
- Hana says "Stiff joints at **fifteen**", but the seller said 13, and the rest of the line is broken.
- Maya walks away over "**a kitten of no known pedigree**", yet the seller said nothing about pedigree and the cat is
  not a kitten: she reacts to something nobody said.
- Leo's line barely makes sense, and he lowers his offer while saying "no health issues".
- Then Leo asks about health again, right after it was answered.

In [5]:
answers.append("Apart from the stiff joints and the FIV result, I do not know of any other problems. She is spayed.")
n = len(m.log)
m.answer_text(answers[1])
show(m, n)
print("anyone left to ask:", m.question)

  You: Apart from the stiff joints and the FIV result, I do not know of any other problems. She is spayed.
Hana: Being spayed is fine. $55.
Leo: No other problems and no other problems in the family, so the missing tests don't change her health. I'll come down to $45.
offers: {'Hana': 55, 'Owen': 0, 'Maya': 0, 'Leo': 45}
anyone left to ask: None


Hana's "Being spayed is fine. $55." is a short stock line, the kind that fills the old training sales.
Leo mentions "no other problems in the family", which the seller never said, and says the answer does not change
her health while still lowering his offer, so the reason and the move disagree. Both pedigree buyers have left and
the other two have asked, so the sale ends after two rounds.

In [6]:
for name in m.present():
    things = m.scene(name)[1:]
    print(name, [o["label"] for o in things], "|", things[0]["reason"])
print(m.sell(max(m.present(), key=m.offers.get)))
print("lines written by the model:", m.stats)

Hana ['picture frame', 'candle', 'chair'] | She'd follow me to my chair by the picture frame of my old girl, near a candle.


Leo ['stethoscope', 'calendar', 'envelope'] | Her vet would check her with a stethoscope and do the tests by syringe, envelope and calendar, the spay date on my calendar.
{'buyer': 'Hana', 'price': 55, 'typical': 75}
lines written by the model: {'lines': 14, 'unreadable': 0}


Hana's scene fits a buyer who lost her old cat, but the line is close to word for word from the training
data (notebook 4 counts "by the picture frame of my old girl" in 14 training targets): it is memorised. Leo's line
mentions a syringe he did not pick, which breaks the scene rule. The cat sells to Hana for $55, below the typical
$75. All 14 lines were readable (`unreadable: 0`), so B1's problem is not the format but the content.

## Side by side on held-out sales

One played sale depends a lot on luck. Here the model writes reactions for situations from the 10 sales B1 never
trained on, next to **what the buyer in the real sale said**. The last line of each example is the program check.

In [7]:
import eval_buyer_b1 as E
from sellanything_ml.buyers.format import END

val = [json.loads(line) for line in open(ML / "data/processed/buyers_b1/val.jsonl")]
reacts = [r for r in val if r["task"] == "react"][::113][:8]          # 8 reactions spread over the set

def complete(gpt, prompt, task, seed):
    return gpt.say(prompt, task, seed)

for i, ex in enumerate(reacts):
    tail = ex["prompt"].split("<|log|>")[-1].replace("<|end|>", "").splitlines()[-2:]
    card = ex["prompt"].split("<|persona|>")[1].split("|")[0]
    out = complete(B1, ex["prompt"], "react", i)
    chk = E.check(ex, out)
    print(f"[{i + 1}] {card.strip()}")
    print("   " + " / ".join(tail))
    print("   sale:", ex["target"].replace(END, ""))
    print("   B1:  ", out)
    print("   check:", "passes" if chk.get("rules_ok") else chk.get("problem"),
          "| same direction" if chk.get("same_direction") else "| other direction")
    print()

[1] Leo, who is retired and home all day
   Ben asked: Can you tell me if the cat has any health issues and if all the vaccinations are up to date? / Seller: Yea, shes FIV pos but shes fully vaccinated and had a vet check last month. She's all up to date on her stuff.<|react|>
   sale: + | Being FIV positive means she stays indoors, which suits me since I'm home all day, and a vet check last month is reassuring. I'll go up to $55. | $55
   B1:   0 | A vet check last month is something I'd gladly do for her. Staying at $50. | $50
   check: passes | other direction



[2] Sam, who already has two dogs
   Elena asked: Can you tell me how independent this cat is? Does she enjoy spending time alone or does she need constant attention? / Seller: This cat is quite independent and enjoys her alone time, but she also loves to play and can be quite vocal at night. She's used to being around other animals, so she doesn't need constant attention from humans to be happy.<|react|>
   sale: 0 | Being used to other animals is good for my dogs. Staying at $70. | $70
   B1:   0 | Being independent is fine with my dogs. Staying at $70. | $70
   check: passes | same direction



[3] Leo, who cares a lot about how things look
   You asked: Can you tell me more about the cat's appearance? Does she have a beautiful coat or any unique features? / Seller: She's a Maine Coon, so she's got a pretty long, fluffy coat. She's still growing into it, but it's getting quite striking. No unique features, just your typical majestic Maine Coon looks.<|react|>
   sale: + | A long, fluffy Maine Coon coat that's getting quite striking is the beautiful coat I want. I'll go up to $80. | $80
   B1:   + | A long, fluffy coat with a fluffy coat look is the striking look I want. I'll go up to $80. | $80
   check: passes | same direction



[4] Ines, who spends weekends hiking and wants an adventure cat
   Leo asked: Can you tell me more about her coat and appearance? / Seller: She's a big Ragdoll, long-haired and pretty, the kind that needs some brushing. Nothing fancy, but you'll want to keep her fur neat.<|react|>
   sale: 0 | Her long hair doesn't matter on hikes. Staying at $75. | $75
   B1:   0 | A long-haired Ragdoll is fine on hikes. Staying at $75. | $75
   check: passes | same direction



[5] Sam, who wants things in excellent shape
   Aisha asked: Can you tell me more about the specific condition of the cat and if there are any ongoing health concerns that might affect her price? / Seller: She's FIV positive, but that just means she needs to stay indoors and get regular check-ups. Her price reflects her condition, but she's still a healthy, calm cat overall. Right now, she's just recovering from a cold, but she's doing better each day.<|react|>
   sale: 0 | Her FIV and cold, as I already counted. Staying at $55. | $55
   B1:   0 | FIV check-ups are fine. Staying at $55. | $55
   check: passes | same direction



[6] Andre, who is buying for a family with young kids
   Ruby asked: How old is she, and how does she act when someone she loves comes home? / Seller: She's ten. Follows you room to room like a dog, and she'll tell you loudly when it's breakfast.<|react|>
   sale: 0 | Following people around might mean she likes my kids, but I'll ask. $55. | $55
   B1:   0 | Following people around is sweet, but my kids are at var. $55. | $55
   check: passes | same direction



[7] Maya, who is a student on a tight budget
   Mei asked: Her coat looks a bit rough in the photo. Is it soft up close? / Seller: its short n a bit scruffy, she was a farm cat. a good brush n it shines up ok<|react|>
   sale: 0 | A good brush costs almost nothing. $25. | $25
   B1:   0 | A good brush costs a bit more. $25. | $25
   check: passes | same direction



[8] Felix, who already has an older cat at home
   Sam asked: How does her coat look at fifteen? / Seller: Her points have darkened with age, and she's a bit thin from the thyroid. Still a lovely face.<|react|>
   sale: 0 | Her lovely face is nice. $45. | $45
   B1:   0 | Her points don't matter to me. $45. | $45
   check: passes | same direction



All 8 examples pass the program rules and 7 move the offer the same way as the buyer in the real sale. The
differences are in the details:
- [1] The real buyer links "FIV positive, so she stays indoors" to being home all day and raises; B1 only says the vet
  check is good and keeps its offer, without tying it to the buyer's life.
- Many lines are short stock lines ("X is fine. $N."), and some contain odd words ([6] "my kids are at var.",
  [3] "fluffy coat with a fluffy coat look").

Side by side is easier than a live sale: these prompts come from well-written sales, while in a live sale one wrong
line leads to the next, so a played sale looks worse than the scores.

## Scores (validation set, program rules)

In [8]:
b1 = json.load(open(ML / "runs/sft_b1/eval_v2_no_budget.json"))["tasks"]
rows = [("ask: new in meaning", "ask", "new_in_meaning"), ("react: passes the rules", "react", "rules_ok"),
        ("react: same direction as the sale", "react", "same_direction"),
        ("react: amount said = offer", "react", "said_equals_offer"),
        ("react: reasonable move", "react", "reasonable"), ("imagine: passes the rules", "imagine", "rules_ok")]
print(f"{'':36}{'B1':>8}")
for label, task, key in rows:
    print(f"{label:36}{b1[task][key]:>8.1%}")

                                          B1
ask: new in meaning                    89.8%
react: passes the rules                90.2%
react: same direction as the sale      78.8%
react: amount said = offer             99.9%
react: reasonable move                 99.9%
imagine: passes the rules              70.8%


B1's scores. When B2 is trained, a column for it goes here and the same cat is played again above, with the
same rule for the seller's answers.